# Microsoft Fabric Notebook Cheatsheet

A runnable reference of common patterns for working with notebooks in Microsoft Fabric — covering Spark/Pandas interop, file I/O, Delta Lake, transformations, SQL, parameters, `notebookutils`, and performance.

Every demo cell creates its own sample data so the notebook runs top to bottom with no external files. Real-world lakehouse paths appear as commented templates next to each demo.

## How to use

1. Download `NB_Fabric_Cheatsheet.ipynb`.
2. In your Fabric workspace: **New → Import notebook → Upload**.
3. Attach any lakehouse (the demos don't depend on its contents).
4. Run cells in order. The final cell drops the demo tables for cleanup.

Built for the **Synapse PySpark** runtime (the Fabric default). `notebookutils` cells are mostly commented — uncomment to execute against your own tenant.

## Contents

| # | Section | Covers |
|---|---------|--------|
| 1 | [Setup & imports](#setup) | Standard imports, runtime info |
| 2 | [DataFrame conversions](#conversions) | Spark ↔ Pandas ↔ Pandas-on-Spark |
| 3 | [Reading files](#read) | CSV, Parquet, Excel, JSON (pandas + Spark) |
| 4 | [Writing files](#write) | Pandas writers, Spark writers, single-file output |
| 5 | [Lakehouse paths](#paths) | Relative / mount / ABFSS, `notebookutils.fs` |
| 6 | [Delta tables](#delta) | `saveAsTable`, `save`, append, CTAS |
| 7 | [Delta operations](#delta-ops) | MERGE, OPTIMIZE, VACUUM, time travel, schema evolution |
| 8 | [Cross-lakehouse access](#cross) | ABFSS + three-part naming |
| 9 | [Transformations](#transforms) | `when`/`otherwise`, rename, cast, dedupe, null handling, group/agg |
| 10 | [SQL in notebooks](#sql) | `%%sql` magic vs `spark.sql()`, temp views |
| 11 | [Parameters](#params) | Parameter cell pattern for pipeline invocation |
| 12 | [`notebookutils`](#nbutils) | `notebook.run`, `credentials`, `runtime`, `lakehouse` |
| 13 | [Logging & error handling](#logging) | Pipeline-friendly try/except pattern |
| 14 | [Performance tips](#perf) | Partitioning, broadcast, cache, V-Order |
| 15 | [Inspect / debug utilities](#utils) | Schema, count, info, dataframe-type check |


---
<a id="setup"></a>

## 1. Setup & imports

Imports pandas, the PySpark functions and types modules, the Delta Lake table API, and the standard date/time types. `spark` and `display()` are pre-defined by the Fabric runtime — no import needed. Running `spark.version` confirms the kernel is alive and shows which Spark build you're on.

In [ ]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType
from delta.tables import DeltaTable
from datetime import datetime, date

# spark and display() are provided by the Fabric runtime
print("Spark version:", spark.version)

---
<a id="conversions"></a>

## 2. DataFrame conversions (Spark ↔ Pandas)

Spark DataFrames are distributed across executors; Pandas DataFrames live in driver memory. Convert carefully — `toPandas()` collects *all* rows to the driver and will OOM on large data.

#### **Spark → Pandas.** `toPandas()` pulls every row of a Spark DataFrame back to the driver and returns a pandas DataFrame. Safe for small result sets (aggregates, samples) — never for raw fact tables.

In [ ]:
spark_df = spark.createDataFrame(
    [(1, "apple", 1.20), (2, "banana", 0.50), (3, "orange", 0.80)],
    ["id", "fruit", "price"],
)

pandas_df = spark_df.toPandas()
display(pandas_df)

#### **Pandas → Spark.** `spark.createDataFrame(pdf)` distributes a driver-side pandas DataFrame across the cluster. Schema is inferred from the pandas dtypes.

In [ ]:
pdf = pandas_df

sdf = spark.createDataFrame(pdf)
display(sdf)

#### **Pandas-on-Spark.** The `pyspark.pandas` module gives you the pandas API but the data stays distributed on the cluster. Use this when you want pandas-style code without collecting to the driver. `.to_spark()` and `.to_pandas()` convert between flavours.

In [ ]:
import pyspark.pandas as ps

psdf = ps.DataFrame({"id": [1, 2, 3], "fruit": ["apple", "banana", "orange"]})
display(psdf.head())

# Convert between the three flavours
sdf_again = psdf.to_spark()     # → Spark DataFrame (distributed)
pdf_again = psdf.to_pandas()    # → Pandas (collects to driver)

---
<a id="read"></a>

## 3. Reading files

Read CSV, Parquet, Excel, JSON using both pandas and Spark. Spark is parallel; pandas is serial. Spark can read directly from ABFSS paths; Pandas typically reads from mounted paths or small extracts.

In [ ]:
# Spark read CSV
# sdf = spark.read.csv("/lakehouse/default/Files/data.csv", header=True, inferSchema=True)

# Spark read Parquet
# sdf = spark.read.parquet("/lakehouse/default/Files/data.parquet")

# Pandas read CSV
# pdf = pd.read_csv("/Volumes/my_volume/data.csv")

# Pandas read Excel
# pdf = pd.read_excel("/Volumes/my_volume/data.xlsx", sheet_name="Sheet1")

print("File read patterns shown as comments above.")

---
<a id="write"></a>

## 4. Writing files

Write Spark DataFrames to Parquet, CSV, or Delta; write Pandas to CSV, Excel, or Parquet. Always use `coalesce(1)` in Spark for single-file output.

In [ ]:
# Spark write (multi-file, distributed)
# sdf.write.mode("overwrite").parquet("/lakehouse/default/Files/output")

# Spark write single file
# sdf.coalesce(1).write.mode("overwrite").csv("/lakehouse/default/Files/output_single.csv", header=True)

# Pandas write
# pdf.to_csv("/Volumes/my_volume/output.csv", index=False)
# pdf.to_excel("/Volumes/my_volume/output.xlsx", sheet_name="Results", index=False)

print("Write patterns shown as comments above.")

---
<a id="paths"></a>

## 5. Lakehouse paths

Files in a lakehouse can be referenced by relative path, OneLake mount, or ABFSS. `notebookutils.fs` is the Python API for filesystem operations.

In [ ]:
# Relative path (Files folder)
# /lakehouse/default/Files/data.csv

# ABFSS path (cross-workspace)
# abfss://workspace_id@onelake.dfs.fabric.microsoft.com/lakehouse_id/Files/data.csv

# List files in Files folder
# files = notebookutils.fs.ls("/lakehouse/default/Files")

# Mkdir
# notebookutils.fs.mkdirs("/lakehouse/default/Files/new_folder")

# Remove file
# notebookutils.fs.rm("/lakehouse/default/Files/old_file.csv", recurse=True)

print("Lakehouse path patterns shown as comments above.")

---
<a id="delta"></a>

## 6. Delta tables

Save a DataFrame as a managed or external Delta table. Managed tables live in the lakehouse metadata; external (CTAS) tables point to ABFSS paths.

In [ ]:
# Create sample dataframe
demo_sdf = spark.createDataFrame(
    [(1, "Alice", 85), (2, "Bob", 90), (3, "Charlie", 78)],
    ["id", "name", "score"]
)

# Write as managed Delta table (metadata in lakehouse)
demo_sdf.write.mode("overwrite").saveAsTable("demo_managed_table")

# Write as external Delta table (data in ABFSS, metadata in lakehouse)
# demo_sdf.write.mode("overwrite").format("delta").save("abfss://workspace@onelake.dfs.fabric.microsoft.com/lh/Tables/demo_external")

# Create as Select (CTAS) — schema is inferred from query
# spark.sql("CREATE TABLE demo_ctas AS SELECT * FROM demo_managed_table WHERE score > 80")

# Append to existing Delta table
more_data = spark.createDataFrame([(4, "Diana", 92)], ["id", "name", "score"])
more_data.write.mode("append").saveAsTable("demo_managed_table")

display(spark.table("demo_managed_table"))

---
<a id="delta-ops"></a>

## 7. Delta operations

MERGE for upserts, OPTIMIZE to compact files, VACUUM to remove old snapshots, time travel to query history, schema evolution for dynamic columns.

In [ ]:
# Read table as Delta object for MERGE
target = DeltaTable.forName(spark, "demo_managed_table")

# Create update DataFrame
updates = spark.createDataFrame(
    [(2, "Bob_Updated", 95), (5, "Eve", 88)],
    ["id", "name", "score"]
)

# MERGE (upsert): match on id, update existing, insert new
target.alias("t").merge(
    updates.alias("u"),
    "t.id = u.id"
).whenMatchedUpdate(set={"t.name": "u.name", "t.score": "u.score"})\
.whenNotMatchedInsert(values={"t.id": "u.id", "t.name": "u.name", "t.score": "u.score"})\
.execute()

# OPTIMIZE: compact small files and reorder
spark.sql("OPTIMIZE demo_managed_table ZORDER BY id")

# VACUUM: remove snapshots older than 7 days (default 30)
# spark.sql("VACUUM demo_managed_table RETAIN 168 HOURS")

# Time travel: query as of a specific version or timestamp
# spark.sql("SELECT * FROM demo_managed_table VERSION AS OF 0").show()

display(spark.table("demo_managed_table"))

---
<a id="cross"></a>

## 8. Cross-lakehouse access

Query tables in other lakehouses using ABFSS paths or three-part naming. Three-part names require the table to be registered in the metastore.

In [ ]:
# Read from another lakehouse using ABFSS
# other_sdf = spark.read.format("delta").load(
#     "abfss://other_ws_id@onelake.dfs.fabric.microsoft.com/other_lh_id/Tables/other_table"
# )

# Three-part naming (if table is registered in the metastore)
# spark.sql("SELECT * FROM other_workspace.other_lakehouse.other_table LIMIT 10").show()

print("Cross-lakehouse access patterns shown as comments above.")

---
<a id="transforms"></a>

## 9. Transformations

Common data manipulation: conditional logic (`when`/`otherwise`), rename, cast, deduplication, null handling, and groupby/aggregations.

In [ ]:
sdf = spark.createDataFrame(
    [(1, "Alice", 85, None), (2, "Bob", 92, "Math"), (3, "Charlie", None, "Science"), (3, "Charlie", 78, "Science")],
    ["id", "name", "score", "subject"]
)

# Conditional logic: when/otherwise
transformed = sdf.withColumn(
    "grade",
    F.when(F.col("score").isNull(), "Incomplete")
     .when(F.col("score") >= 90, "A")
     .when(F.col("score") >= 80, "B")
     .otherwise("C")
)

# Rename column
transformed = transformed.withColumnRenamed("subject", "department")

# Cast type
transformed = transformed.withColumn("id_str", F.col("id").cast("string"))

# Deduplication
deduped = transformed.dropDuplicates(["id", "name"])

# Fill nulls
filled = deduped.fillna({"score": 0, "department": "Unknown"})

# GroupBy and aggregate
agg = sdf.groupBy("name").agg(
    F.avg("score").alias("avg_score"),
    F.count("id").alias("count")
)

display(agg)

---
<a id="sql"></a>

## 10. SQL in notebooks

Use `%%sql` magic cells for SQL queries (renders nicely in notebooks) or `spark.sql()` for inline SQL in Python. Both have pros and cons.

In [ ]:
# Python approach: spark.sql() returns a DataFrame
result = spark.sql("""
    SELECT id, name, score,
           CASE WHEN score >= 90 THEN 'A'
                WHEN score >= 80 THEN 'B'
                ELSE 'C' END as grade
    FROM demo_managed_table
    WHERE score IS NOT NULL
""")

display(result)

# Create a temporary view from a DataFrame for SQL access
sdf.createOrReplaceTempView("temp_scores")
spark.sql("SELECT * FROM temp_scores WHERE score > 80").show()

---
<a id="params"></a>

## 11. Parameters

Mark a cell with the `parameters` tag for pipeline invocation. Fabric pipeline can override these values at runtime.

In [ ]:
# Parameters — Fabric pipeline can override these
TABLE_NAME = "demo_managed_table"
MIN_SCORE = 80
OUTPUT_TABLE = "filtered_results"

print(f"Processing table: {TABLE_NAME}, filtering for scores >= {MIN_SCORE}")

---
<a id="nbutils"></a>

## 12. `notebookutils`

Fabric-specific API for notebook orchestration, credentials, runtime info, and lakehouse access.

In [ ]:
# Run another notebook
# notebookutils.notebook.run("/path/to/other_notebook", 30, {"param1": "value1"})

# Get current context
# context = notebookutils.notebook.getContext()

# Get runtime info
# print(notebookutils.runtime.getContext())

# Exit notebook with status
# notebookutils.notebook.exit("Completed successfully")

print("notebookutils patterns shown as comments above. Uncomment to use.")

---
<a id="logging"></a>

## 13. Logging & error handling

Pipeline-friendly pattern: try/except, log errors, exit with status.

In [ ]:
from datetime import datetime

try:
    start = datetime.now()
    
    # Your code here
    sdf = spark.table("demo_managed_table")
    row_count = sdf.count()
    
    elapsed = (datetime.now() - start).total_seconds()
    print(f"✓ Processed {row_count} rows in {elapsed:.2f}s")
    
except FileNotFoundError as e:
    print(f"✗ File not found: {e}")
    # notebookutils.notebook.exit(f"Error: {e}")
    
except Exception as e:
    print(f"✗ Unexpected error: {type(e).__name__}: {e}")
    # notebookutils.notebook.exit(f"Error: {e}")

---
<a id="perf"></a>

## 14. Performance tips

Partitioning, broadcast joins, caching, and V-Order for faster queries.

In [ ]:
# Partitioning: write in partitions to speed up queries on subset
# demo_sdf.write.partitionBy("subject").mode("overwrite").saveAsTable("partitioned_table")

# Broadcast join: small table → all executors (no shuffle)
# df_big.join(F.broadcast(df_small), on="key")

# Cache (in memory for repeated access)
# sdf.cache()  # consume one action to materialize
# sdf.count()

# V-Order: write in Fabric-optimized format (auto on Delta tables)
# spark.sql("OPTIMIZE demo_managed_table ZORDER BY id")

# Repartition for parallelism
# sdf.repartition(8).write.mode("overwrite").saveAsTable("repartitioned_table")

print("Performance patterns shown as comments above.")

---
<a id="utils"></a>

## 15. Inspect / debug utilities

Schema, row count, info, and type checking for quick exploration.

In [ ]:
sdf = spark.table("demo_managed_table")

# Schema
print("Schema:")
sdf.printSchema()

# Row count
print(f"\nRow count: {sdf.count()}")

# Info (like pandas)
print(f"\nColumns: {sdf.columns}")
print(f"Partitions: {sdf.rdd.getNumPartitions()}")

# Type check
print(f"\nType: {type(sdf)}")
print(f"Is DataFrame? {isinstance(sdf, type(spark.createDataFrame([], 'x INT')))}")

# Sample
print("\nSample (first 3 rows):")
display(sdf.limit(3))

---

## Cleanup

Drop all demo tables to clean up.

In [ ]:
# Drop demo tables
spark.sql("DROP TABLE IF EXISTS demo_managed_table")
print("✓ Cleaned up demo tables")